# MFVI predictive-variance — full reproducibility gate\n\nUpload the supplied `RG7maF4bGu-colab-bundle.tar.gz`, then run every cell in order. This notebook runs the unmodified pinned author implementation on all nine cached UCI datasets, an independent NumPy/SciPy audit, and the fail-closed local publication gate.

In [ ]:
from google.colab import files\nfrom pathlib import Path\nimport shutil, subprocess\n\nuploaded = files.upload()\narchive = next((Path(name) for name in uploaded if name.endswith('.tar.gz')), None)\nif archive is None:\n    raise RuntimeError('Upload RG7maF4bGu-colab-bundle.tar.gz')\nsubprocess.run(['tar', '-tzf', str(archive)], check=True, stdout=subprocess.DEVNULL)\nsubprocess.run(['tar', '-xzf', str(archive), '-C', '/content'], check=True)\nPROJECT = Path('/content/icml26-repro-RG7maF4bGu-mfvi-predictive-variance')\nif not PROJECT.is_dir():\n    raise RuntimeError(f'expected project directory missing: {PROJECT}')\n%cd /content/icml26-repro-RG7maF4bGu-mfvi-predictive-variance\nprint('Input bundle extracted:', PROJECT)

In [ ]:
import os, sys\n\n# Colab normally provides Torch globally. Retain system packages in the local\n# venv, then add only the verifier/gate dependencies at pinned-compatible ranges.\nsubprocess.run([sys.executable, '-m', 'venv', '.venv', '--system-site-packages'], check=True)\nvenv_python = str(PROJECT / '.venv/bin/python')\nsubprocess.run([venv_python, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'], check=True)\nsubprocess.run([venv_python, '-m', 'pip', 'install', '--quiet', '-r', 'repro/requirements.txt'], check=True)\ntorch_check = subprocess.run([venv_python, '-c', 'import torch; print(torch.__version__)'])\nif torch_check.returncode != 0:\n    subprocess.run([venv_python, '-m', 'pip', 'install', '--quiet', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch>=2.5,<2.7'], check=True)\nsubprocess.run([venv_python, '-c', 'import torch, pandas, numpy, scipy; print({"torch": torch.__version__, "cuda": torch.cuda.is_available()})'], check=True)

In [ ]:
# Cheap preflight: validates the exact source/data pins and independent controls.\nenvironment = {**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}\nsubprocess.run([venv_python, 'repro/src/verify_mfvi.py', '--mode', 'synthetic', '--output', 'outputs/colab_synthetic_preflight.json'], check=True, env=environment)\nsubprocess.run(['bash', 'repro/src/run_full_gate.sh'], check=True, env=environment)\nsubprocess.run([venv_python, '-m', 'pytest', '-q', 'repro/tests'], check=True, env=environment)\nprint((PROJECT / 'outputs/prepublish_gate.json').read_text())

In [ ]:
# Package only the raw evidence required for independent readback.\nresult_archive = Path('/content/RG7maF4bGu-colab-results.tar.gz')\nif result_archive.exists():\n    result_archive.unlink()\nsubprocess.run(['tar', '-czf', str(result_archive), '-C', str(PROJECT), 'outputs'], check=True)\nsubprocess.run(['sha256sum', str(result_archive)], check=True)\nfiles.download(str(result_archive))